# 시도별 차로당 통행량 점수화

`input`에 통행량 파일과 차로 파일을 넣고 아래 파일명을 바꾼 뒤 **Run → Run All Cells**를 실행하세요.

**점수 = (도로별 최대 all_aadt / lane) / (같은 시도의 최대 차로당 통행량) × 100**, 소수점 한 자리.

도로별 최대값은 같은 `sido_code`, `lev5_5_link_id`의 입력 행에서 구합니다. 동일 기준연도·방향·집계 조건을 사용하세요.
차로 파일의 `lev5_5_link_id`는 통행량과 같은 Level 5.5 ID여야 합니다. 표준노드링크 LINK_ID를 그대로 넣으면 안 됩니다.


In [ ]:
# 패키지가 없다면 다음 줄의 주석을 해제하고 실행하세요.
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
import pandas as pd
pd.options.display.float_format = "{:.1f}".format
from score_aadt import load_data, score_aadt, save_results

INPUT_PATH = Path("input/traffic.csv")  # 본인 데이터 파일명으로 변경
LANE_PATH = Path("input/lanes.csv")  # 컬럼: lev5_5_link_id, lane
# 통행량 파일 자체에 lane 컬럼이 있으면 LANE_PATH = None
CSV_ENCODING = "utf-8-sig"  # 한글 CSV 인코딩 오류 시 "cp949"
OUTPUT_DIR = Path("output")


In [ ]:
data = load_data(INPUT_PATH, encoding=CSV_ENCODING)
print(f"입력 행 수: {len(data):,}")
lane_data = load_data(LANE_PATH, encoding=CSV_ENCODING) if LANE_PATH is not None else None
result, summary = score_aadt(data, lane_data)
display(result.head(20))


In [ ]:
display(summary)
print("행 처리 결과:")
display(result["score_status"].value_counts(dropna=False).rename_axis("score_status").to_frame("row_count"))
positive = summary["sido_max_per_lane_aadt"].gt(0).fillna(False)
assert summary.loc[positive, "max_score"].eq(100.0).all(), "시도 최대점수 검증 실패"
assert result["traffic_score"].dropna().between(0, 100).all(), "점수 범위 검증 실패"
print("검증 완료: 양의 최대 차로당 통행량이 있는 시도의 최고 점수는 100.0입니다.")


In [ ]:
result_path = save_results(result, OUTPUT_DIR)
print(f"반출 파일: {result_path.resolve()}")
print("반출 컬럼: lev5_5_link_id, traffic_score (소수점 한 자리)")
print("점수 계산이 안 된 링크는 제외하고, 같은 링크는 한 번만 저장합니다.")


`traffic_score`가 최종 점수입니다. `road_max_aadt`는 도로별 최대 AADT, `per_lane_aadt`는 차로당 통행량, `sido_max_per_lane_aadt`는 시도 최대 차로당 통행량입니다.

차로 수 누락·잘못된 차로 수·유효 AADT가 없는 도로·시도/링크 코드 누락은 점수가 비어 있습니다. `score_status`를 확인하세요. 차로 수가 없는 도로는 시도 최대값 계산에서도 제외됩니다.
한 도로의 일부 AADT가 잘못돼도 다른 유효 행이 있으면 그 도로의 유효 최댓값을 사용합니다. 원본 행 순서는 유지됩니다.
시도 차로당 통행량이 전부 0이면 0.0점입니다. 원본은 수정하지 않고 결과 파일은 재실행 시 갱신합니다.

반출 CSV는 `lev5_5_link_id`, `traffic_score` 두 컬럼만 저장합니다. 통행량·차로 수·지역 코드·시도별 요약은 파일로 저장하지 않습니다.
